# Lab 2 - Transformers from Scratch in PyTorch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hyperscaleailabs/ml-platform-engineering/blob/main/notebooks/02_transformer_pytorch.ipynb)

**Runtime:** ~10 minutes on a Colab T4, ~15 on CPU. One small text download (1.1 MB) with an
offline fallback.

Lab 1 ended with an MLP: a fixed-size input mapped to a fixed-size output. Language is neither.
This lab builds a decoder-only transformer that consumes variable-length sequences, one component
at a time, and **verifies every component against a reference implementation** before assembling it.

The architecture is deliberately the modern one - RMSNorm, RoPE, grouped-query attention, SwiGLU,
tied embeddings. That is not a stylistic choice: it is the Qwen2/Qwen3 block, so the model you
write here is structurally the model you fine-tune with LoRA in Labs 3 and 4.

## What you will build

| Section | Component | Verified against |
|---|---|---|
| 2 | Scaled dot-product attention | `F.scaled_dot_product_attention` |
| 3 | Causal masking | An explicit loop over positions |
| 4 | Multi-head + grouped-query attention | Per-head reference computation |
| 5 | RoPE | The relative-position identity it is supposed to satisfy |
| 6 | RMSNorm, pre-norm residual stream | `nn.LayerNorm` and a gradient-norm experiment |
| 7 | SwiGLU feed-forward | Parameter-matched GELU MLP |
| 8-9 | Full model, trained | A bigram baseline and a token-frequency floor |
| 10 | Sampling: greedy / temperature / top-k / top-p | - |
| 11 | KV cache | Bit-comparable logits, then a wall-clock benchmark |

**Prerequisites:** Lab 1, or equivalent comfort with `nn.Module`, autograd, and a training loop.

## 0. Setup

In [ ]:
import os
import sys
import math
import time
import urllib.request
from dataclasses import dataclass, field

IN_COLAB = "google.colab" in sys.modules

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ---- Config (override via environment variables) -------------------------------------------------
SEED = int(os.environ.get("LAB_SEED", 0))
MAX_STEPS = int(os.environ.get("LAB_MAX_STEPS", 1500))   # training steps for the small LM
EVAL_EVERY = int(os.environ.get("LAB_EVAL_EVERY", 150))
BATCH_SIZE = int(os.environ.get("LAB_BATCH_SIZE", 32))
BLOCK_SIZE = int(os.environ.get("LAB_BLOCK_SIZE", 128))  # context length in tokens (characters)
# --------------------------------------------------------------------------------------------------


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = pick_device()


def set_seed(seed: int = SEED) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def check(condition: bool, message: str) -> None:
    if not condition:
        raise AssertionError(f"CHECK FAILED: {message}")
    print(f"  ok - {message}")


set_seed()
print(f"torch  {torch.__version__}")
print(f"device {DEVICE}")

## 1. Data and tokenization

We use **character-level** tokenization. Real models use byte-pair encoding (Lab 3 uses Qwen's
151k-token BPE vocabulary), but characters remove a moving part: a ~65-symbol vocabulary means the
embedding table is negligible and every effect you measure comes from the transformer itself.

The tokenizer contract is the same at any granularity:
`encode: str -> list[int]`, `decode: list[int] -> str`, and `decode(encode(s)) == s`.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = "tinyshakespeare.txt"

FALLBACK_TEXT = (
    "To be, or not to be, that is the question:\n"
    "Whether 'tis nobler in the mind to suffer\n"
    "The slings and arrows of outrageous fortune,\n"
    "Or to take arms against a sea of troubles\n"
    "And by opposing end them.\n"
) * 400  # only used if the download fails; enough structure to make the loss move


def load_corpus() -> str:
    if os.path.exists(DATA_PATH):
        with open(DATA_PATH, "r", encoding="utf-8") as f:
            return f.read()
    try:
        urllib.request.urlretrieve(DATA_URL, DATA_PATH)
        with open(DATA_PATH, "r", encoding="utf-8") as f:
            return f.read()
    except Exception as exc:  # offline / firewalled environments
        print(f"download failed ({exc}); using the small embedded fallback corpus")
        return FALLBACK_TEXT


text = load_corpus()
vocab = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(vocab)}
itos = {i: ch for ch, i in stoi.items()}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[int(i)] for i in ids)

VOCAB_SIZE = len(vocab)
print(f"corpus {len(text):,} characters, vocabulary {VOCAB_SIZE} symbols")
print(f"vocabulary: {''.join(vocab)!r}")
print(f"\nsample:\n{text[:200]}")

check(decode(encode("Hello, world!")) == "Hello, world!", "encode/decode round-trips")

In [ ]:
# Split before doing anything else. A 90/10 split by position is fine for a single continuous
# corpus; for a document collection you would split by document to avoid leaking near-duplicates.
data = torch.tensor(encode(text), dtype=torch.long)
n_train = int(0.9 * len(data))
train_data, val_data = data[:n_train], data[n_train:]
print(f"train {len(train_data):,} tokens   val {len(val_data):,} tokens")


def get_batch(split: str, batch_size: int = BATCH_SIZE, block_size: int = BLOCK_SIZE):
    """Sample `batch_size` random windows of length `block_size`.

    `y` is `x` shifted by one: at every position the target is the *next* token. One sequence of
    length T therefore supplies T training signals, not one - this density is why next-token
    prediction is such an efficient objective.
    """
    src = train_data if split == "train" else val_data
    ix = torch.randint(len(src) - block_size - 1, (batch_size,))
    x = torch.stack([src[i:i + block_size] for i in ix])
    y = torch.stack([src[i + 1:i + 1 + block_size] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)


xb, yb = get_batch("train", batch_size=2, block_size=8)
print(f"\nx {tuple(xb.shape)}  y {tuple(yb.shape)}")
print(f"x[0] = {decode(xb[0])!r}")
print(f"y[0] = {decode(yb[0])!r}   <- shifted by exactly one character")
check(torch.equal(xb[0, 1:], yb[0, :-1]), "targets are inputs shifted left by one")

## 2. Scaled dot-product attention

Attention is a differentiable, content-addressed lookup. Each position emits a **query** (what am I
looking for), a **key** (what I offer), and a **value** (what I pass on if selected):

$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

### Why divide by $\sqrt{d_k}$

For $q, k$ with i.i.d. zero-mean unit-variance components, $q\cdot k$ has variance $d_k$. Feed
scores of magnitude $\sqrt{d_k}\approx 11$ (for $d_k=128$) into a softmax and it saturates: one
weight goes to ~1, the rest to ~0, and the gradient through the softmax vanishes. The scaling keeps
the score variance at 1 regardless of head dimension. We measure both effects below.

In [ ]:
set_seed()
d_k = 128
q_probe, k_probe = torch.randn(4096, d_k), torch.randn(4096, d_k)
raw = (q_probe * k_probe).sum(-1)
print(f"var(q.k)            {raw.var():.1f}   (theory: d_k = {d_k})")
print(f"var(q.k / sqrt(d_k)) {(raw / math.sqrt(d_k)).var():.3f}   (theory: 1.0)")

scores_unscaled = q_probe[:8] @ k_probe[:8].T
for name, s in [("unscaled", scores_unscaled), ("scaled", scores_unscaled / math.sqrt(d_k))]:
    p = s.softmax(-1)
    entropy = -(p * p.clamp_min(1e-9).log()).sum(-1).mean()
    print(f"{name:>9}: max softmax weight {p.max():.4f}, mean entropy {entropy:.4f} nats "
          f"(uniform would be {math.log(8):.4f})")

check(abs(raw.var().item() - d_k) / d_k < 0.1, "the dot product variance really is ~d_k")
check((scores_unscaled.softmax(-1).max() > (scores_unscaled / math.sqrt(d_k)).softmax(-1).max()),
      "unscaled scores saturate the softmax")

In [ ]:
def attention_from_scratch(q, k, v, mask=None, dropout_p=0.0, training=False):
    """Scaled dot-product attention.

    Shapes: q (..., Tq, hd), k/v (..., Tk, hd), mask broadcastable to (..., Tq, Tk),
    where True means "this position may be attended to".
    """
    d_head = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_head)      # (..., Tq, Tk)
    if mask is not None:
        # -inf before the softmax, not 0 after it: zeroing post-softmax would leave the
        # denominator polluted by masked positions and the weights would not sum to 1.
        scores = scores.masked_fill(~mask, float("-inf"))
    weights = scores.softmax(dim=-1)
    if dropout_p > 0 and training:
        weights = F.dropout(weights, p=dropout_p)
    return weights @ v, weights


set_seed()
B, H, T, HD = 2, 4, 16, 32
q = torch.randn(B, H, T, HD)
k = torch.randn(B, H, T, HD)
v = torch.randn(B, H, T, HD)

mine, attn_w = attention_from_scratch(q, k, v)
reference = F.scaled_dot_product_attention(q, k, v)      # PyTorch's fused/flash kernel

print(f"output {tuple(mine.shape)}   max |mine - reference| = {(mine - reference).abs().max():.2e}")
check(torch.allclose(mine, reference, atol=1e-5), "matches F.scaled_dot_product_attention")
check(torch.allclose(attn_w.sum(-1), torch.ones(B, H, T), atol=1e-5), "attention weights sum to 1 per query")

> **In production you call `F.scaled_dot_product_attention`**, which dispatches to FlashAttention.
> Flash never materializes the $(T,T)$ score matrix: it tiles the computation and keeps running
> softmax statistics, turning attention's memory cost from $O(T^2)$ to $O(T)$. Same math, same
> numbers (as just verified), radically different memory profile - which is the entire reason
> 128k-token contexts are practical.

## 3. Causal masking

A language model must not see the future. Position $t$ may attend to $[0, t]$ only - otherwise the
answer leaks into the input and validation loss collapses to near zero while generation produces
garbage. This is the single most common silent bug when writing a transformer from scratch, so we
verify it two ways: structurally, and by comparing against an explicit loop.

In [ ]:
causal_mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
masked_out, masked_w = attention_from_scratch(q, k, v, mask=causal_mask)

print("attention weights for the first 6 positions (head 0, batch 0):")
print(np.round(masked_w[0, 0, :6, :6].numpy(), 3))

# Independent check: recompute position t using only k/v[:t+1] with no mask at all.
loop_out = torch.stack([
    attention_from_scratch(q[:, :, t:t + 1], k[:, :, :t + 1], v[:, :, :t + 1])[0].squeeze(2)
    for t in range(T)
], dim=2)

check(torch.allclose(masked_out, loop_out, atol=1e-5), "masked attention equals an explicit causal loop")
check(torch.allclose(masked_w[0, 0, 0, 1:], torch.zeros(T - 1), atol=1e-7),
      "position 0 attends only to itself")
check((masked_w.triu(diagonal=1).abs().max() == 0), "no weight above the diagonal anywhere")
check(torch.allclose(F.scaled_dot_product_attention(q, k, v, is_causal=True), masked_out, atol=1e-5),
      "matches F.scaled_dot_product_attention(is_causal=True)")

## 4. Multi-head and grouped-query attention

One attention head can express one relation per position. Multi-head attention splits `d_model`
into `n_heads` independent subspaces of size `d_model / n_heads` - same FLOPs, several relations
learned in parallel.

**Grouped-query attention (GQA)** keeps `n_heads` query heads but only `n_kv_heads` key/value
heads, each shared by a group of queries. The motivation is not training compute - it is the KV
cache. During generation you store K and V for every past token:

$$\text{cache bytes} = 2 \cdot n_{\text{layers}} \cdot n_{kv} \cdot d_{\text{head}} \cdot T \cdot B \cdot \text{sizeof(dtype)}$$

That term, not the weights, is what caps batch size at long context. Qwen2.5-0.5B uses 14 query
heads and 2 KV heads - a 7x cache reduction. You will watch that cache dominate memory in Lab 3.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Causal multi-head attention with optional grouped-query sharing and a KV cache."""

    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int | None = None, dropout: float = 0.0):
        super().__init__()
        n_kv_heads = n_kv_heads or n_heads
        assert d_model % n_heads == 0, "d_model must divide evenly into heads"
        assert n_heads % n_kv_heads == 0, "each KV head must serve a whole number of query heads"

        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads          # queries per KV head
        self.d_head = d_model // n_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.o_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)

    def forward(self, x, rope=None, kv_cache=None, return_weights=False):
        B, T, _ = x.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)     # (B, Hq, T, hd)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)  # (B, Hkv, T, hd)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        # Rotary embeddings are applied to q and k only (never v), at their absolute positions.
        pos_offset = 0 if kv_cache is None else kv_cache.length
        if rope is not None:
            q, k = rope(q, k, offset=pos_offset)

        if kv_cache is not None:
            k, v = kv_cache.append(k, v)     # k/v now span all positions seen so far

        # Expand KV heads to match query heads. repeat_interleave (not repeat) so that
        # query heads [0..n_rep) share KV head 0, [n_rep..2*n_rep) share KV head 1, and so on.
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        weights = None
        if return_weights:
            Tk = k.size(2)
            mask = torch.ones(T, Tk, dtype=torch.bool, device=x.device).tril(diagonal=Tk - T)
            out, weights = attention_from_scratch(q, k, v, mask=mask,
                                                  dropout_p=self.dropout, training=self.training)
        else:
            # is_causal=True is only correct when q and k have equal length. During cached
            # decoding the single new query legitimately attends to every cached key.
            is_causal = kv_cache is None or T > 1
            out = F.scaled_dot_product_attention(
                q, k, v, is_causal=is_causal,
                dropout_p=self.dropout if self.training else 0.0,
            )

        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return (self.o_proj(out), weights) if return_weights else (self.o_proj(out), None)


# Verify GQA head sharing explicitly: query head h must use KV head h // n_rep.
set_seed()
mha = MultiHeadAttention(d_model=64, n_heads=8, n_kv_heads=2).eval()
x_probe = torch.randn(1, 12, 64)
with torch.no_grad():
    fused, _ = mha(x_probe)
    fused_w, w = mha(x_probe, return_weights=True)

    B_, T_ = x_probe.shape[0], x_probe.shape[1]
    qh = mha.q_proj(x_probe).view(B_, T_, 8, 8).transpose(1, 2)
    kh = mha.k_proj(x_probe).view(B_, T_, 2, 8).transpose(1, 2)
    vh = mha.v_proj(x_probe).view(B_, T_, 2, 8).transpose(1, 2)
    heads = [attention_from_scratch(qh[:, h], kh[:, h // 4], vh[:, h // 4],
                                    mask=torch.tril(torch.ones(T_, T_, dtype=torch.bool)))[0]
             for h in range(8)]
    manual = mha.o_proj(torch.cat(heads, dim=-1))

print(f"fused vs per-head reference: {(fused - manual).abs().max():.2e}")
check(torch.allclose(fused, manual, atol=1e-5), "GQA shares KV heads exactly as specified")
check(torch.allclose(fused, fused_w, atol=1e-5), "the explicit-weights path agrees with the fused kernel")
check(mha.k_proj.weight.shape[0] == 16, "with 2 KV heads the K projection is 4x smaller than Q's")

## 5. Position information: why you need it, and RoPE

Attention as defined so far is **permutation-equivariant**: shuffle the tokens and the outputs
shuffle with them, unchanged. "dog bites man" and "man bites dog" would be indistinguishable.
Demonstrate it rather than take it on faith:

In [ ]:
set_seed()
x_seq = torch.randn(1, 6, 64)
perm = torch.tensor([3, 1, 0, 5, 2, 4])
no_pos = MultiHeadAttention(64, 4).eval()
with torch.no_grad():
    # No causal mask here - the mask itself injects order, which would confound the test.
    q_ = no_pos.q_proj(x_seq).view(1, 6, 4, 16).transpose(1, 2)
    k_ = no_pos.k_proj(x_seq).view(1, 6, 4, 16).transpose(1, 2)
    v_ = no_pos.v_proj(x_seq).view(1, 6, 4, 16).transpose(1, 2)
    out_a = attention_from_scratch(q_, k_, v_)[0]
    out_b = attention_from_scratch(q_[:, :, perm], k_[:, :, perm], v_[:, :, perm])[0]

check(torch.allclose(out_a[:, :, perm], out_b, atol=1e-5),
      "attention without positional information is exactly permutation-equivariant")

### Rotary Position Embedding (RoPE)

RoPE rotates $q$ and $k$ in 2-D subspaces by an angle proportional to absolute position. Because
rotations compose, the dot product between a query at position $m$ and a key at position $n$ ends
up depending **only on $m-n$**:

$$\langle R_m q,\; R_n k \rangle = \langle R_{m-n} q,\; k \rangle$$

So you get relative-position behaviour without any relative-position bookkeeping, no learned
position table, and the ability to extrapolate past the training length (by interpolating or
rescaling $\theta$ - the basis of YaRN and NTK scaling). This is what Llama, Qwen, Mistral and
essentially every current open model use. The identity is the specification, so we test it directly.

In [ ]:
def rotate_half(x):
    """Split the head dim in half and rotate: [x1, x2] -> [-x2, x1].

    This is the HuggingFace/Llama convention (halves), not the interleaved-pairs convention from
    the original paper. They are related by a permutation of head channels and are equivalent as
    long as you are consistent - but load weights trained under the other convention and the model
    silently degrades. Labs 3 and 4 rely on this being the HF convention.
    """
    d = x.shape[-1] // 2
    return torch.cat((-x[..., d:], x[..., :d]), dim=-1)


class RotaryEmbedding(nn.Module):
    def __init__(self, d_head: int, max_seq: int = 4096, theta: float = 10_000.0):
        super().__init__()
        assert d_head % 2 == 0, "RoPE needs an even head dimension"
        inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2, dtype=torch.float32) / d_head))
        t = torch.arange(max_seq, dtype=torch.float32)
        freqs = torch.outer(t, inv_freq)                 # (max_seq, d_head/2)
        emb = torch.cat((freqs, freqs), dim=-1)          # duplicated to match rotate_half
        self.register_buffer("cos", emb.cos(), persistent=False)
        self.register_buffer("sin", emb.sin(), persistent=False)
        self.max_seq = max_seq

    def forward(self, q, k, offset: int = 0):
        T = q.size(-2)
        assert offset + T <= self.max_seq, f"position {offset + T} exceeds the RoPE cache ({self.max_seq})"
        cos = self.cos[offset:offset + T].to(q.dtype)     # (T, d_head)
        sin = self.sin[offset:offset + T].to(q.dtype)
        return (q * cos + rotate_half(q) * sin,
                k * cos + rotate_half(k) * sin)


set_seed()
rope = RotaryEmbedding(d_head=32, max_seq=256)

# To test the identity we need the SAME q and k vectors placed at different absolute positions -
# so broadcast one random vector across all 256 slots and let RoPE rotate each copy by its index.
q_one, k_one = torch.randn(32), torch.randn(32)
q_r = q_one.expand(1, 1, 256, 32).contiguous()
k_r = k_one.expand(1, 1, 256, 32).contiguous()
qr, kr = rope(q_r, k_r)

score = lambda m, n: (qr[0, 0, m] * kr[0, 0, n]).sum()
print(f"unrotated q.k   = {(q_one * k_one).sum():+.6f}   (offset 0)")
print(f"score(  0,   0) = {score(0, 0):+.6f}")
print(f"score( 10,   5) = {score(10, 5):+.6f}")
print(f"score(105, 100) = {score(105, 100):+.6f}   <- far apart in absolute position...")
print(f"score(200, 195) = {score(200, 195):+.6f}   <- ...but the same relative offset of 5")
print(f"score( 10,   4) = {score(10, 4):+.6f}   <- offset 6, genuinely different")

check(torch.allclose(score(10, 5), score(105, 100), atol=1e-3)
      and torch.allclose(score(10, 5), score(200, 195), atol=1e-3),
      "RoPE scores depend only on relative distance, not absolute position")
check(not torch.allclose(score(10, 5), score(10, 4), atol=1e-3),
      "...while a different distance gives a different score")
check(torch.allclose(score(0, 0), (q_one * k_one).sum(), atol=1e-4),
      "at zero offset RoPE leaves the dot product unchanged")
check(torch.allclose(qr.norm(dim=-1), q_r.norm(dim=-1), atol=1e-4),
      "RoPE is a rotation - it preserves vector norms")

In [ ]:
# Visualize the rotation frequencies. Low channel indices rotate fast (they encode local, token-level
# offsets); high indices rotate slowly (they stay near-constant over thousands of tokens and encode
# coarse, long-range position). Raising theta stretches every wavelength - that is exactly how
# context-length extension works.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].imshow(rope.cos[:128, :16].T, aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
axes[0].set_xlabel("position"); axes[0].set_ylabel("channel pair"); axes[0].set_title("cos(m * inv_freq)")
for ch in [0, 2, 5, 10, 15]:
    axes[1].plot(rope.cos[:128, ch], label=f"channel {ch}")
axes[1].set_xlabel("position"); axes[1].set_title("Fast channels are local, slow channels are global")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 6. Normalization and the residual stream

### RMSNorm

LayerNorm centres and rescales: $\frac{x-\mu}{\sigma}\gamma + \beta$. RMSNorm drops the mean
subtraction and the bias:

$$\mathrm{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum_i x_i^2 + \epsilon}} \odot \gamma$$

It works about as well, costs one fewer reduction pass, and has half the parameters - which is why
every recent architecture uses it.

### Pre-norm, not post-norm

The residual stream must stay an unobstructed identity path: `x + f(norm(x))`, not `norm(x + f(x))`.

The difference is structural. In pre-norm, $\frac{\partial x_{\ell+1}}{\partial x_\ell} = I + \dots$ -
there is an exact identity term, so gradient flows to layer 0 no matter how deep the stack is. In
post-norm, every layer's output passes *through* a normalization on the way forward, so the
backward pass is rescaled at every step and the input is effectively decoupled from the loss. The
measurement below shows the size of that effect, and one consequence people miss.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        # Reduce in float32 even when the activations are bf16: the sum of squares is exactly the
        # kind of reduction that loses precision in low bit-widths.
        dtype = x.dtype
        x = x.float()
        x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.to(dtype)) * self.weight


set_seed()
rms = RMSNorm(64)
x_n = torch.randn(4, 10, 64) * 5 + 2
out_rms = rms(x_n)
print(f"input  RMS {x_n.pow(2).mean(-1).sqrt().mean():.3f}")
print(f"output RMS {out_rms.pow(2).mean(-1).sqrt().mean():.3f}")
check(torch.allclose(out_rms.pow(2).mean(-1).sqrt(), torch.ones(4, 10), atol=1e-3),
      "RMSNorm produces unit root-mean-square activations")
check(sum(p.numel() for p in rms.parameters()) == 64
      and sum(p.numel() for p in nn.LayerNorm(64).parameters()) == 128,
      "RMSNorm has half the parameters of LayerNorm (no bias term)")

In [ ]:
# Two identical stacks, differing only in where the norm sits. We measure the quantity that
# actually matters: how much gradient survives the trip back to the *input* of the stack.
class ToyBlock(nn.Module):
    def __init__(self, d, pre_norm=True):
        super().__init__()
        self.norm, self.lin, self.pre_norm = nn.LayerNorm(d), nn.Linear(d, d), pre_norm

    def forward(self, x):
        return x + self.lin(self.norm(x)) if self.pre_norm else self.norm(x + self.lin(x))


depths = [2, 8, 24, 48, 96]
flow = {"pre-norm": {"grad": [], "rms": []}, "post-norm": {"grad": [], "rms": []}}
for depth in depths:
    for pre in (True, False):
        set_seed()
        stack = nn.Sequential(*[ToyBlock(64, pre) for _ in range(depth)])
        inp = torch.randn(8, 64, requires_grad=True)
        out = stack(inp)
        out.pow(2).mean().backward()
        key = "pre-norm" if pre else "post-norm"
        flow[key]["grad"].append(inp.grad.norm().item())
        flow[key]["rms"].append(out.detach().pow(2).mean().sqrt().item())

print(f"{'depth':>6} {'pre dL/dx':>12} {'post dL/dx':>12} {'pre out RMS':>12} {'post out RMS':>13}")
for i, d_ in enumerate(depths):
    print(f"{d_:>6} {flow['pre-norm']['grad'][i]:>12.3e} {flow['post-norm']['grad'][i]:>12.3e} "
          f"{flow['pre-norm']['rms'][i]:>12.2f} {flow['post-norm']['rms'][i]:>13.2f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for name in flow:
    axes[0].plot(depths, flow[name]["grad"], "o-", label=name)
    axes[1].plot(depths, flow[name]["rms"], "o-", label=name)
axes[0].set_yscale("log"); axes[0].set_xlabel("depth"); axes[0].set_ylabel("||dL/dx_input||")
axes[0].set_title("Gradient reaching the stack input"); axes[0].legend()
axes[1].set_xlabel("depth"); axes[1].set_ylabel("output RMS")
axes[1].set_title("Residual stream magnitude"); axes[1].legend()
plt.tight_layout(); plt.show()

ratio = flow["pre-norm"]["grad"][-1] / flow["post-norm"]["grad"][-1]
print(f"\nat depth {depths[-1]}, pre-norm delivers {ratio:.0f}x more gradient to the input")
check(all(p > 100 * q for p, q in zip(flow["pre-norm"]["grad"], flow["post-norm"]["grad"])),
      "pre-norm delivers orders of magnitude more gradient to the input, at every depth")
check(flow["pre-norm"]["rms"][-1] > 2 * flow["pre-norm"]["rms"][0],
      "the pre-norm residual stream grows with depth")
check(abs(flow["post-norm"]["rms"][-1] - 1.0) < 0.05,
      "the post-norm stream is pinned to unit RMS - the normalization erases the scale, "
      "and with it the gradient signal")

The right-hand plot is the consequence people miss: **pre-norm's residual stream grows with
depth**, because every block adds to it and nothing ever rescales it. Post-norm pins it to unit
RMS. Two things in the model below exist purely to manage that growth:

* a **final norm** before the LM head, so the output projection sees a stable scale;
* initializing the residual-path output projections (`o_proj`, `down_proj`) with std scaled by
  $1/\sqrt{2L}$, so the accumulated variance stays bounded as layers are added.

Post-norm is not unusable - the original 2017 transformer used it - but it needs careful learning
rate warmup to train at all past a dozen layers. Pre-norm plus these two adjustments is what made
100-layer stacks routine.

## 7. The feed-forward block: SwiGLU

Attention moves information *between* positions; the feed-forward network transforms it *within* a
position. It holds roughly two thirds of a transformer's parameters.

The classic form is `W2(act(W1 x))` with hidden size $4d$. SwiGLU splits the up-projection into a
value path and a **gate**:

$$\mathrm{SwiGLU}(x) = W_{\text{down}}\big(\mathrm{SiLU}(W_{\text{gate}}x) \odot W_{\text{up}}x\big)$$

Three matrices instead of two, so the hidden size is scaled to $\tfrac{8}{3}d$ to keep the
parameter count roughly matched. The multiplicative gate lets a unit suppress its own output
conditionally - a strictly larger function class than a pointwise activation, and empirically worth
it at equal parameters.

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d_model: int, d_hidden: int | None = None, multiple_of: int = 32):
        super().__init__()
        if d_hidden is None:
            d_hidden = int(8 * d_model / 3)
            d_hidden = multiple_of * ((d_hidden + multiple_of - 1) // multiple_of)  # keep GEMMs aligned
        self.gate_proj = nn.Linear(d_model, d_hidden, bias=False)
        self.up_proj = nn.Linear(d_model, d_hidden, bias=False)
        self.down_proj = nn.Linear(d_hidden, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


d_demo = 256
swiglu = SwiGLU(d_demo)
gelu_mlp = nn.Sequential(nn.Linear(d_demo, 4 * d_demo, bias=False), nn.GELU(),
                         nn.Linear(4 * d_demo, d_demo, bias=False))
p_swiglu = sum(p.numel() for p in swiglu.parameters())
p_gelu = sum(p.numel() for p in gelu_mlp.parameters())
print(f"SwiGLU (hidden {swiglu.gate_proj.out_features}): {p_swiglu:,} params")
print(f"GELU   (hidden {4 * d_demo}): {p_gelu:,} params")
print(f"ratio {p_swiglu / p_gelu:.2f}")
check(0.85 < p_swiglu / p_gelu < 1.15, "the 8/3 rule keeps SwiGLU parameter-matched to a 4d GELU MLP")

## 8. Assembling the model

Every component is verified, so we can stack them. The block is:

```
x = x + attention(rmsnorm(x))
x = x + swiglu(rmsnorm(x))
```

Plus one detail that matters at this scale: **tied embeddings**. The input embedding table and the
output projection share weights. At `d_model=192` and 65 characters that saves little, but
Qwen2.5-0.5B has a 151,936-token vocabulary and a hidden size of 896 - the embedding table is
136M parameters, over a quarter of the model. Tying is why "0.5B" is 0.49B and not 0.63B.

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = VOCAB_SIZE
    block_size: int = BLOCK_SIZE
    n_layers: int = 4
    n_heads: int = 6
    n_kv_heads: int = 2
    d_model: int = 192
    dropout: float = 0.1
    tie_embeddings: bool = True
    rope_theta: float = 10_000.0


class KVCache:
    """Per-layer key/value cache for incremental decoding."""

    def __init__(self):
        self.k = None
        self.v = None

    @property
    def length(self) -> int:
        return 0 if self.k is None else self.k.size(2)

    def append(self, k, v):
        # A production cache preallocates to max length and writes in place; concatenating
        # reallocates every step. Kept simple here so the mechanism stays visible.
        self.k = k if self.k is None else torch.cat([self.k, k], dim=2)
        self.v = v if self.v is None else torch.cat([self.v, v], dim=2)
        return self.k, self.v

    def nbytes(self) -> int:
        return 0 if self.k is None else self.k.numel() * self.k.element_size() * 2


class TransformerBlock(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model)
        self.attn = MultiHeadAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.dropout)
        self.ffn_norm = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg.d_model)
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, rope=None, kv_cache=None, return_weights=False):
        attn_out, weights = self.attn(self.attn_norm(x), rope=rope, kv_cache=kv_cache,
                                      return_weights=return_weights)
        x = x + self.resid_dropout(attn_out)
        x = x + self.resid_dropout(self.ffn(self.ffn_norm(x)))
        return x, weights


class GPT(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.rope = RotaryEmbedding(cfg.d_model // cfg.n_heads,
                                    max_seq=max(4 * cfg.block_size, 1024), theta=cfg.rope_theta)

        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight   # one tensor, two uses

        self.apply(self._init_weights)
        # Scale the residual-path output projections by 1/sqrt(2*n_layers) so the residual stream
        # variance does not grow with depth (GPT-2's trick, still standard).
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, kv_caches=None, return_weights=False):
        x = self.drop(self.tok_emb(idx))
        all_weights = []
        for i, block in enumerate(self.blocks):
            x, w = block(x, rope=self.rope,
                         kv_cache=None if kv_caches is None else kv_caches[i],
                         return_weights=return_weights)
            if return_weights:
                all_weights.append(w)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, all_weights

    def num_params(self, non_embedding: bool = False) -> int:
        n = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n -= self.tok_emb.weight.numel()
        return n

    def new_caches(self):
        return [KVCache() for _ in self.blocks]


set_seed()
cfg = GPTConfig()
model = GPT(cfg).to(DEVICE)
print(f"parameters: {model.num_params():,} total, {model.num_params(True):,} excluding embeddings")

logits, loss, _ = model(*get_batch("train"))
print(f"logits {tuple(logits.shape)}   initial loss {loss.item():.4f}")
print(f"uniform-guess loss = ln(vocab) = {math.log(VOCAB_SIZE):.4f}")

check(logits.shape == (BATCH_SIZE, BLOCK_SIZE, VOCAB_SIZE), "logits are (batch, time, vocab)")
check(abs(loss.item() - math.log(VOCAB_SIZE)) < 0.35,
      "an untrained model sits at the uniform-distribution loss - the init is sane")
check(model.lm_head.weight.data_ptr() == model.tok_emb.weight.data_ptr(),
      "embeddings are tied: input and output share one tensor")

In [ ]:
# Causality check on the real model. Perturb the LAST token; every earlier position's logits must be
# byte-identical. If this fails, the model is cheating and every metric downstream is meaningless.
model.eval()
with torch.no_grad():
    probe = get_batch("val", batch_size=1)[0]
    base, _, _ = model(probe)
    tampered = probe.clone()
    tampered[0, -1] = (tampered[0, -1] + 1) % VOCAB_SIZE
    after, _, _ = model(tampered)

check(torch.equal(base[0, :-1], after[0, :-1]), "changing the last token leaves all earlier logits identical")
check(not torch.equal(base[0, -1], after[0, -1]), "...while the last position does change")

## 9. Training

The loop is Lab 1's, plus three things that matter once models get deep:

* **AdamW with decoupled weight decay**, applied to matrices but *not* to norms, biases, or
  embeddings - decaying a norm's gain fights the normalization itself.
* **Cosine schedule with linear warmup.** Adam's second-moment estimate is unreliable for the first
  few hundred steps; a large step then can knock the model into a bad region it never leaves.
* **Gradient clipping** by global norm, a cheap insurance policy against one pathological batch.

A useful sanity number: training FLOPs are approximately $6ND$ (6 per parameter per token: two for
the forward multiply-accumulate, four for the backward). We print the estimate below.

In [ ]:
def configure_optimizer(model, lr=3e-4, weight_decay=0.1, betas=(0.9, 0.95)):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (decay if p.dim() >= 2 else no_decay).append(p)
    print(f"weight decay on {len(decay)} tensors ({sum(p.numel() for p in decay):,} params), "
          f"none on {len(no_decay)} ({sum(p.numel() for p in no_decay):,})")
    return torch.optim.AdamW(
        [{"params": decay, "weight_decay": weight_decay},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr, betas=betas,
    )


def lr_at(step, max_steps, base_lr=3e-4, warmup_frac=0.05, min_frac=0.1):
    warmup = max(1, int(warmup_frac * max_steps))
    if step < warmup:
        return base_lr * (step + 1) / warmup
    progress = (step - warmup) / max(1, max_steps - warmup)
    return base_lr * (min_frac + (1 - min_frac) * 0.5 * (1 + math.cos(math.pi * progress)))


@torch.no_grad()
def estimate_loss(model, iters=20):
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = [model(*get_batch(split))[1].item() for _ in range(iters)]
        out[split] = float(np.mean(losses))
    model.train()
    return out


def train_lm(model, max_steps=MAX_STEPS, base_lr=3e-4, log_every=EVAL_EVERY, label=""):
    opt = configure_optimizer(model, lr=base_lr)
    hist = {"step": [], "train": [], "val": [], "lr": []}
    model.train()
    t0 = time.time()

    for step in range(max_steps):
        lr = lr_at(step, max_steps, base_lr)
        for g in opt.param_groups:
            g["lr"] = lr

        xb, yb = get_batch("train")
        _, loss, _ = model(xb, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if step % log_every == 0 or step == max_steps - 1:
            e = estimate_loss(model)
            hist["step"].append(step); hist["train"].append(e["train"])
            hist["val"].append(e["val"]); hist["lr"].append(lr)
            print(f"{label}step {step:5d}  train {e['train']:.4f}  val {e['val']:.4f}  "
                  f"lr {lr:.2e}  {time.time() - t0:5.1f}s")

    tokens = max_steps * BATCH_SIZE * BLOCK_SIZE
    flops = 6 * model.num_params() * tokens
    print(f"{label}{tokens:,} tokens seen, ~{flops / 1e12:.2f} TFLOPs of compute, "
          f"{time.time() - t0:.1f}s wall clock")
    return hist


set_seed()
model = GPT(cfg).to(DEVICE)
history = train_lm(model)

### Is the loss actually good?

A number like 1.7 means nothing on its own. Two reference points make it interpretable:

* **Uniform** over the vocabulary: `ln(65) = 4.17` nats. Anything at or above this is broken.
* **Unigram** (predict the training-set character frequencies, ignoring context): the entropy of
  the marginal distribution. Beating this proves the model uses context at all.

Perplexity is `exp(loss)` - the effective number of characters the model is choosing among. In
character-LM literature you will also see bits-per-character, which is `loss / ln(2)`.

In [ ]:
counts = torch.bincount(train_data, minlength=VOCAB_SIZE).float()
unigram_p = counts / counts.sum()
unigram_loss = -(unigram_p * unigram_p.clamp_min(1e-12).log()).sum().item()
final_val = history["val"][-1]

print(f"uniform baseline    {math.log(VOCAB_SIZE):.4f} nats   ppl {VOCAB_SIZE:8.1f}")
print(f"unigram baseline    {unigram_loss:.4f} nats   ppl {math.exp(unigram_loss):8.1f}")
print(f"transformer (val)   {final_val:.4f} nats   ppl {math.exp(final_val):8.1f}   "
      f"bpc {final_val / math.log(2):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(history["step"], history["train"], "o-", ms=3, label="train")
axes[0].plot(history["step"], history["val"], "o-", ms=3, label="val")
axes[0].axhline(unigram_loss, color="grey", ls=":", label="unigram baseline")
axes[0].axhline(math.log(VOCAB_SIZE), color="red", ls=":", label="uniform baseline")
axes[0].set_xlabel("step"); axes[0].set_ylabel("cross-entropy (nats)"); axes[0].legend(fontsize=8)
axes[0].set_title("Training curve against baselines")
axes[1].plot(history["step"], history["lr"], "o-", ms=3)
axes[1].set_xlabel("step"); axes[1].set_ylabel("learning rate"); axes[1].set_title("Warmup + cosine decay")
plt.tight_layout(); plt.show()

check(final_val < unigram_loss, f"the model beats the context-free unigram baseline "
                                f"({final_val:.3f} < {unigram_loss:.3f})")

### What the heads learned

Attention maps are suggestive rather than conclusive - do not over-read them - but at this scale
the structure is legible: some heads attend to the immediately preceding token (an induction-like
local pattern), others spread attention broadly or park it on a few high-frequency delimiters.

In [ ]:
model.eval()
sample_text = text[1000:1000 + 64]
sample_ids = torch.tensor([encode(sample_text)], device=DEVICE)
with torch.no_grad():
    _, _, ws = model(sample_ids, return_weights=True)

layer = len(ws) - 1
fig, axes = plt.subplots(1, min(4, cfg.n_heads), figsize=(14, 3.4))
for h, ax in enumerate(np.atleast_1d(axes)):
    ax.imshow(ws[layer][0, h].cpu().numpy(), cmap="viridis", aspect="auto")
    ax.set_title(f"layer {layer}, head {h}", fontsize=9)
    ax.set_xlabel("attends to"); ax.set_ylabel("query position" if h == 0 else "")
plt.suptitle(f"Attention over {sample_text[:40]!r}...", fontsize=10)
plt.tight_layout(); plt.show()

# The lower-triangular structure is a second, independent confirmation of causality.
check(float(ws[layer][0].triu(diagonal=1).abs().max()) == 0.0,
      "the trained model's attention is strictly lower-triangular")

## 10. Sampling

Generation is a loop: forward, take the last position's logits, sample, append. The sampling rule
is a product decision, not a model property:

| Strategy | Effect | Failure mode |
|---|---|---|
| Greedy (`argmax`) | deterministic | repetition loops |
| Temperature $\tau$ | $\tau<1$ sharpens, $\tau>1$ flattens | high $\tau$ produces noise |
| Top-k | sample from the k most likely | fixed k ignores how peaked the distribution is |
| Top-p (nucleus) | smallest set with cumulative mass $\ge p$ | adapts k per step - usually the best default |

In [ ]:
@torch.no_grad()
def generate(model, prompt: str, max_new_tokens: int = 200, temperature: float = 1.0,
             top_k: int | None = None, top_p: float | None = None, use_cache: bool = True,
             seed: int | None = None):
    if seed is not None:
        torch.manual_seed(seed)
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)
    caches = model.new_caches() if use_cache else None

    for step in range(max_new_tokens):
        if use_cache:
            # Only the newest token needs a forward pass - everything else is in the cache.
            step_input = idx if step == 0 else idx[:, -1:]
            logits, _, _ = model(step_input, kv_caches=caches)
        else:
            logits, _, _ = model(idx[:, -model.cfg.block_size:])

        logits = logits[:, -1, :]
        if temperature == 0.0:
            next_id = logits.argmax(-1, keepdim=True)
        else:
            logits = logits / temperature
            if top_k is not None:
                kth = torch.topk(logits, min(top_k, logits.size(-1)), dim=-1).values[:, -1:]
                logits = logits.masked_fill(logits < kth, float("-inf"))
            if top_p is not None:
                sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
                cum = sorted_logits.softmax(-1).cumsum(-1)
                remove = cum - sorted_logits.softmax(-1) > top_p     # keep the token that crosses p
                logits = logits.masked_fill(remove.scatter(1, sorted_idx, remove), float("-inf"))
            next_id = torch.multinomial(logits.softmax(-1), num_samples=1)

        idx = torch.cat([idx, next_id], dim=1)
    return decode(idx[0].tolist())


prompt = "ROMEO:"
for name, kwargs in [
    ("greedy (temp=0)", dict(temperature=0.0)),
    ("temp=0.8, top_k=20", dict(temperature=0.8, top_k=20)),
    ("temp=1.0, top_p=0.9", dict(temperature=1.0, top_p=0.9)),
    ("temp=2.0 (too hot)", dict(temperature=2.0)),
]:
    print(f"\n{'=' * 78}\n{name}\n{'=' * 78}")
    print(generate(model, prompt, max_new_tokens=180, seed=SEED, **kwargs))

In [ ]:
# Greedy decoding is deterministic; sampling is not. Worth verifying, because "my generations are
# not reproducible" is usually a missing seed rather than a real bug.
g1 = generate(model, prompt, max_new_tokens=40, temperature=0.0)
g2 = generate(model, prompt, max_new_tokens=40, temperature=0.0)
s1 = generate(model, prompt, max_new_tokens=40, temperature=1.0, seed=1)
s2 = generate(model, prompt, max_new_tokens=40, temperature=1.0, seed=2)
check(g1 == g2, "greedy decoding is deterministic")
check(s1 != s2, "sampling with different seeds gives different text")

## 11. The KV cache

Naive generation re-runs the whole prefix for every new token: generating $T$ tokens costs
$O(T^2)$ forward passes' worth of work, and it recomputes keys and values that cannot have changed
(causal masking means position $t$'s K/V never depend on anything after $t$).

The KV cache stores them. Each step then processes exactly one token. The cost is memory, and it
grows linearly with sequence length and batch size - which is why GQA exists, why paged attention
(vLLM) exists, and why serving throughput is usually a memory problem rather than a compute one.

**First correctness, then speed** - a fast cache that changes the output is worthless.

In [ ]:
model.eval()
test_prompt = text[:24]
ids = torch.tensor([encode(test_prompt)], device=DEVICE)

with torch.no_grad():
    full_logits, _, _ = model(ids)                       # one pass over the whole prefix

    caches = model.new_caches()
    step_logits = []
    for t in range(ids.size(1)):                         # one token at a time, through the cache
        out, _, _ = model(ids[:, t:t + 1], kv_caches=caches)
        step_logits.append(out[:, -1])
    step_logits = torch.stack(step_logits, dim=1)

max_dev = (full_logits - step_logits).abs().max().item()
print(f"max |full-pass logits - cached-decode logits| = {max_dev:.2e}")
print(f"cache size after {ids.size(1)} tokens: {sum(c.nbytes() for c in caches) / 1024:.1f} KiB")

check(max_dev < 2e-4, "cached incremental decoding reproduces the full forward pass")

### How much work does the cache actually save?

Measure the *work*, not the clock. Token-positions pushed through the transformer is a
hardware-independent quantity, and it is what the asymptotics are about:

* **cached:** exactly 1 position per step -> $T$ in total.
* **uncached:** the whole prefix every step -> $\sum_t \min(t, \text{block\_size})$, which is
  quadratic in $T$ until it hits the context limit and linear (with a large constant) after.

In [ ]:
def positions_processed(n_tokens: int, prompt_len: int, block_size: int, use_cache: bool) -> int:
    if use_cache:
        return prompt_len + n_tokens - 1     # the prompt in one pass, then one token per step
    return sum(min(prompt_len + t, block_size) for t in range(n_tokens))


prompt_len = len(encode("ROMEO:"))
lengths = [32, 64, 128, 256, 512]
work = [(n,
         positions_processed(n, prompt_len, cfg.block_size, False),
         positions_processed(n, prompt_len, cfg.block_size, True)) for n in lengths]

print(f"{'tokens':>7} {'no cache':>12} {'cached':>8} {'work ratio':>12}")
for n, w_plain, w_cached in work:
    print(f"{n:>7} {w_plain:>12,} {w_cached:>8,} {w_plain / w_cached:>11.1f}x")

check(work[-1][1] / work[-1][2] > work[0][1] / work[0][2],
      "the work the cache saves grows with generation length")
check(work[0][2] == prompt_len + lengths[0] - 1, "cached decoding processes one position per step")

### Why the wall clock will not show that ratio here

A 128x reduction in arithmetic should be a large speedup. At this scale it is not, and it is worth
understanding exactly why before you trust any inference benchmark.

This model is ~1.5M parameters with `d_model=192`. Its per-step GEMMs are far too small to
saturate a GPU or even a modern CPU vector unit, so runtime is dominated by fixed per-call
overhead - Python dispatch, kernel launch, allocator traffic - which both strategies pay once per
generated token. The probe below measures that directly.

In [ ]:
@torch.no_grad()
def time_forward(n_tokens: int, repeats: int = 20) -> float:
    x = torch.randint(0, VOCAB_SIZE, (1, n_tokens), device=DEVICE)
    for _ in range(5):
        model(x)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(repeats):
        model(x)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / repeats


t_one, t_many = time_forward(1), time_forward(cfg.block_size)
print(f"forward on   1 token  : {t_one * 1e3:6.3f} ms")
print(f"forward on {cfg.block_size:3d} tokens: {t_many * 1e3:6.3f} ms")
print(f"cost ratio {t_many / t_one:.2f}x for {cfg.block_size}x the arithmetic")
print("\nA ratio near 1 means this model is overhead-bound, not compute-bound: the extra 127")
print("token-positions are effectively free, so eliminating them cannot speed anything up.")

In [ ]:
# The wall clock, reported honestly. Expect the two to be close at this scale, and expect noise -
# use the minimum over repeats, which is the standard estimator for a latency floor (the mean is
# contaminated by scheduler noise, and there is no such thing as a spuriously *fast* run).
def bench_generate(use_cache: bool, n_tokens: int, repeats: int = 5) -> float:
    with torch.no_grad():
        generate(model, "R", max_new_tokens=8, use_cache=use_cache)          # warm up
        times = []
        for _ in range(repeats):
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            generate(model, "ROMEO:", max_new_tokens=n_tokens, use_cache=use_cache, temperature=0.0)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    return min(times)


bench_lengths = [64, 128, 256]
rows = []
for n in bench_lengths:
    tc, tp = bench_generate(True, n), bench_generate(False, n)
    rows.append((n, tp, tc, tp / tc))

print(f"{'tokens':>7} {'no cache (s)':>13} {'cached (s)':>11} {'speedup':>8} {'tok/s cached':>13}")
for n, tp, tc, sp in rows:
    print(f"{n:>7} {tp:>13.3f} {tc:>11.3f} {sp:>7.2f}x {n / tc:>13.1f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot([w[0] for w in work], [w[1] for w in work], "o-", label="no cache")
axes[0].plot([w[0] for w in work], [w[2] for w in work], "o-", label="KV cache")
axes[0].set_xlabel("tokens generated"); axes[0].set_ylabel("token-positions processed")
axes[0].set_title("Work: the asymptotics"); axes[0].legend()
axes[1].plot(bench_lengths, [r[1] for r in rows], "o-", label="no cache")
axes[1].plot(bench_lengths, [r[2] for r in rows], "o-", label="KV cache")
axes[1].set_xlabel("tokens generated"); axes[1].set_ylabel("seconds")
axes[1].set_title("Wall clock: overhead-bound at 1.5M params"); axes[1].legend()
plt.tight_layout(); plt.show()

check(t_many / t_one < 10, "this model really is overhead-bound - the benchmark above is expected "
                           "to under-report the cache's value")

**This is the lesson, not a disappointment.** A microbenchmark at the wrong scale can show the
exact opposite of the asymptotic truth. Three rules follow:

1. **Measure work and time.** Work tells you the scaling; time tells you where you sit on the
   roofline today. Only one of them transfers to a different machine.
2. **Benchmark at deployment scale.** Qwen2.5-0.5B has `d_model=896` and 24 layers - roughly 300x
   this model's arithmetic per token, comfortably compute-bound, where the cache is the difference
   between usable and unusable. You will see exactly that in Lab 3.
3. **Batch changes the answer again.** At batch 64 the per-step GEMMs grow 64x while the fixed
   overhead stays put, so the same code moves from overhead-bound to compute-bound. Continuous
   batching in vLLM and TGI exists precisely to keep serving on that side of the line.

In [ ]:
# What a real serving deployment has to budget for. This is the calculation that decides your
# maximum batch size - and note that it is independent of how many parameters the model has.
def kv_cache_bytes(n_layers, n_kv_heads, d_head, seq_len, batch, bytes_per_elem=2):
    return 2 * n_layers * n_kv_heads * d_head * seq_len * batch * bytes_per_elem


print("KV cache footprint in bf16 (2 bytes/element):\n")
print(f"{'model':<28} {'ctx':>7} {'batch':>6} {'cache':>12}")
for name, (L, Hkv, hd) in {
    "this lab's model": (cfg.n_layers, cfg.n_kv_heads, cfg.d_model // cfg.n_heads),
    "Qwen2.5-0.5B (14q/2kv)": (24, 2, 64),
    "Qwen2.5-7B (28q/4kv)": (28, 4, 128),
    "same 7B without GQA": (28, 28, 128),
}.items():
    for ctx, bs in [(2048, 1), (32768, 8)]:
        gb = kv_cache_bytes(L, Hkv, hd, ctx, bs) / 1e9
        print(f"{name:<28} {ctx:>7} {bs:>6} {gb:>10.3f} GB")
    print()

## 12. Exercises

1. **Break causality.** Delete the causal mask and retrain for 200 steps. Validation loss will
   plummet. Then generate text and explain, in one sentence, why the loss became meaningless.
2. **Scaling.** Train `n_layers` in `{2, 4, 8}` at fixed step count and plot final validation loss
   against parameter count on log axes. Does it look like a power law? What confounds this
   experiment at a fixed step budget?
3. **RoPE theta.** Train at `rope_theta=10_000`, then evaluate on sequences twice `block_size`
   (the RoPE cache already allows it). Retrain with `rope_theta=100_000` and compare the
   extrapolation gap. This is the core idea behind long-context extension.
4. **GQA ablation.** Sweep `n_kv_heads` over `{1, 2, 3, 6}` at fixed `n_heads=6`. Plot validation
   loss against the KV cache size from section 11. Where is the knee?
5. **Weight tying.** Set `tie_embeddings=False`. Compare parameter count, final loss, and the
   cosine similarity between `tok_emb.weight` and `lm_head.weight` after training - how close does
   the untied model come to rediscovering the tie on its own?
6. **A real tokenizer.** Swap the character vocabulary for `tiktoken`'s `gpt2` encoding. Sequence
   length in *characters* jumps ~4x for the same token budget. Compare bits-per-character (not
   per-token, which is not comparable across tokenizers).

Exercise 1 is worked below, because "my validation loss is suspiciously good" is a real failure
mode and you should know exactly what it looks like.

In [ ]:
# Solution to exercise 1: what a leaky (non-causal) model looks like.
class LeakyBlock(TransformerBlock):
    def forward(self, x, rope=None, kv_cache=None, return_weights=False):
        h = self.attn_norm(x)
        B, T, _ = h.shape
        a = self.attn
        q = a.q_proj(h).view(B, T, a.n_heads, a.d_head).transpose(1, 2)
        k = a.k_proj(h).view(B, T, a.n_kv_heads, a.d_head).transpose(1, 2)
        v = a.v_proj(h).view(B, T, a.n_kv_heads, a.d_head).transpose(1, 2)
        if rope is not None:
            q, k = rope(q, k)
        if a.n_rep > 1:
            k, v = k.repeat_interleave(a.n_rep, 1), v.repeat_interleave(a.n_rep, 1)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=False)   # <-- the bug
        out = a.o_proj(out.transpose(1, 2).contiguous().view(B, T, -1))
        x = x + self.resid_dropout(out)
        return x + self.resid_dropout(self.ffn(self.ffn_norm(x))), None


set_seed()
leaky = GPT(cfg).to(DEVICE)
leaky.blocks = nn.ModuleList([LeakyBlock(cfg) for _ in range(cfg.n_layers)]).to(DEVICE)
leaky_hist = train_lm(leaky, label="[leaky] ")   # identical budget to the causal model

print(f"\ncausal model   val loss {final_val:.4f}   ppl {math.exp(final_val):.1f}")
print(f"leaky  model   val loss {leaky_hist['val'][-1]:.4f}   ppl {math.exp(leaky_hist['val'][-1]):.1f}")
print("\ntext generated by the leaky model (it can only cheat when the future is given to it,")
print("and at generation time it is not, so the sampled text is far worse than the loss implies):")
print(generate(leaky, "ROMEO:", max_new_tokens=120, temperature=0.8, top_k=20,
               use_cache=False, seed=SEED))

check(leaky_hist["val"][-1] < final_val,
      "at an identical training budget the leaky model reports a BETTER validation loss "
      "while generating worse text - always suspect a leak when a metric beats expectations")

## What you built, and what comes next

* Scaled dot-product attention, verified numerically against PyTorch's fused kernel, plus the
  variance argument for the $1/\sqrt{d_k}$ scale.
* Causal masking, verified three independent ways - including via the failure mode when it is absent.
* Grouped-query attention, and the KV-cache arithmetic that motivates it.
* RoPE, verified against the relative-position identity that defines it.
* RMSNorm, pre-norm residuals (with the gradient-flow measurement), and SwiGLU.
* A trained character LM measured against real baselines, four sampling strategies, and a KV cache
  that is correct before it is fast.

That block - RMSNorm + RoPE + GQA + SwiGLU + tied embeddings - **is** the Qwen2 block. In
**Lab 3** you stop writing it and start adapting one: load Qwen2.5-0.5B from HuggingFace, attach
LoRA adapters to the exact `q_proj`/`k_proj`/`v_proj`/`o_proj` modules you just implemented, and
train, evaluate, and benchmark it. **Lab 4** then reimplements that same forward pass in JAX,
checks it against the HuggingFace logits, and scales the workload out with Ray.